# ML Modeler + ML Reviewer — Step 4 Review

Human-run notebook for the **Step 4: Prompts + Router** slice.

Plan: `project_planning/sean_step_artifacts/ML_Modeler_Reviewer_Implementation_Plan.md`  
Checklist: `project_planning/sean_step_artifacts/ML_Modeler_Reviewer_Checklist.md`

Step 4 adds one new router function plus the supporting config and state keys:
- `orchestration/router.py :: route_after_modeling_review` — decides whether to loop the modeling phase back to its modeler node or advance to the next phase.
- `orchestration/state.py :: modeling_iteration: int` — counter incremented by the 5 reviewed modeler modes (baseline, tune, adjust_lr, feature_selection, final_recommendation).
- `config/workflows.yaml :: workflows.modeling.max_iterations` — the cap consulted by the router.
- `_MODELING_REVIEW_NEXT_NODE` mapping in `router.py` — the advance target per reviewed phase.

Run top to bottom to walk every router branch and confirm the iteration cap fires.

In [ ]:
from pathlib import Path
import subprocess
from pprint import pprint

from multi_agent_ds.orchestration.router import (
    _MODELING_REVIEW_NEXT_NODE,
    _modeling_iteration_limit,
    route_after_modeling_review,
)
from multi_agent_ds.orchestration.state import PipelineState

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the repo root from the current notebook working directory.")

ROOT = resolve_repo_root()
print("Repo root:", ROOT)
print("Configured modeling iteration cap:", _modeling_iteration_limit())
print("\nPhase → advance target map:")
pprint(_MODELING_REVIEW_NEXT_NODE)
print("\nmodeling_iteration present in PipelineState:",
      "modeling_iteration" in PipelineState.__annotations__)

def run_pytest(args):
    cmd = ["uv", "run", "pytest", *args]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f"pytest failed with exit code {completed.returncode}")


## 1. Accept path — each reviewed phase advances to the correct next node

When the reviewer sets `should_revise_modeling=False`, the router must advance to the next phase in the 8-phase modeling flow.

In [ ]:
accept_cases = [
    ("baseline", "ml_modeler_n_estimator_search"),
    ("tune", "ml_modeler_train_tuned"),
    ("adjust_lr", "ml_modeler_importance_review"),
    ("feature_selection", "ml_modeler_final_recommendation"),
    ("final_recommendation", "end"),
]

print(f"{'current_phase':<22} {'router verdict':<36} {'expected':<36}")
print("-" * 96)
for phase, expected in accept_cases:
    state = {"current_phase": phase, "should_revise_modeling": False, "modeling_iteration": 1}
    actual = route_after_modeling_review(state)
    match = "OK" if actual == expected else "MISMATCH"
    print(f"{phase:<22} {actual:<36} {expected:<36} {match}")


## 2. Revise path under the iteration cap — loops back to the modeler

When `should_revise_modeling=True` AND `modeling_iteration < cap`, the router returns the loop-back node `ml_modeler_<phase>`.

In [ ]:
cap = _modeling_iteration_limit()
print(f"Iteration cap: {cap}. Using iteration = 1 (well under the cap).\n")

for phase in ("baseline", "tune", "adjust_lr", "feature_selection", "final_recommendation"):
    state = {"current_phase": phase, "should_revise_modeling": True, "modeling_iteration": 1}
    actual = route_after_modeling_review(state)
    expected = f"ml_modeler_{phase}"
    match = "OK" if actual == expected else "MISMATCH"
    print(f"  phase={phase:<22} -> {actual:<36} (expected {expected}) {match}")


## 3. Revise path with cap exhausted — forces advance

Even with `should_revise_modeling=True`, once `modeling_iteration >= cap` the router must advance to the next phase. This prevents infinite revision loops.

In [ ]:
cap = _modeling_iteration_limit()
print(f"Iteration cap: {cap}. Using iteration = {cap} (at the cap).\n")

cap_cases = [
    ("baseline", "ml_modeler_n_estimator_search"),
    ("tune", "ml_modeler_train_tuned"),
    ("adjust_lr", "ml_modeler_importance_review"),
    ("feature_selection", "ml_modeler_final_recommendation"),
    ("final_recommendation", "end"),
]

for phase, expected in cap_cases:
    state = {"current_phase": phase, "should_revise_modeling": True, "modeling_iteration": cap}
    actual = route_after_modeling_review(state)
    match = "OK" if actual == expected else "MISMATCH"
    print(f"  phase={phase:<22} -> {actual:<36} (expected {expected}) {match}")


## 4. Unknown current_phase — router raises

If the graph ever calls this router with an unexpected `current_phase`, we want a loud failure, not a silent misroute.

In [ ]:
try:
    route_after_modeling_review({"current_phase": "bogus", "should_revise_modeling": False})
except ValueError as exc:
    print(f"Raised as expected: {exc}")
else:
    print("FAILURE: router silently accepted an unknown phase")


## 5. Regression tests

Run the focused router + modeler + reviewer + earlier-step tests in a clean subprocess.
Expected: **47 passed** (8 router tests incl. 7 new modeling-review tests, 17 modeler/reviewer tests, 22 earlier-step tests).

In [ ]:
run_pytest([
    "tests/test_langgraph_router.py",
    "tests/test_pre_modeling_review_agents.py",
    "tests/test_cleaning.py",
    "tests/test_feature_engineering.py",
    "tests/test_preparation_workflow.py",
])


## 6. Review sign-off

If you are satisfied:
- tick the two Human review checkpoint boxes for Step 4 in `ML_Modeler_Reviewer_Checklist.md`
- then we commit Step 4 and move to Step 5 (Final validation).

**Per-phase iteration semantics:** `modeling_iteration` resets to 1 whenever the modeler enters a new phase (fresh entry) and increments only on loop-back (reviewer revise verdict sent control back to the same phase). This means burning the cap on `baseline` does not starve `tune` of its own revision budget — each reviewed phase gets up to `workflows.modeling.max_iterations` attempts. See `ml_modeler._next_modeling_iteration`.

**Known deferred items for a later slice:**
- The graph.py wiring (`add_conditional_edges(ml_reviewer_baseline_review, route_after_modeling_review)` etc.) is not part of this slice. Router correctness is exercised via unit tests that hand-craft the state.
- The modeler does not yet *react* to a reviewer critique on re-run. Looping back currently re-executes the same prompt; plumbing the reviewer's revision_questions into the modeler's next-turn prompt is a follow-up.

Flag anything else you want adjusted before Step 5.